# 07 Non-residual MoE Theory Loops

This notebook intentionally leaves the checkpoint/residual family.
The goal is to mine recent MoE ideas from LLM, vision, retrieval, and
dynamic-compute papers and translate them into DQA hypotheses.

The output is a fifteen-loop screening table and a selected next
non-residual full-design candidate.

## Research Seeds

- DeepSeek / auxiliary-loss-free load balancing
- DeepSeekMoE fine-grained expert segmentation
- BASE / Expert Choice balanced assignment
- GRIN gradient-informed routing
- Mixture-of-Depths and Router-Tuning dynamic compute
- CartesianMoE factorized routing
- V-MoE adaptive per-image compute
- RouterRetriever / routing consistency ideas

In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

cwd = Path.cwd().resolve()
if cwd.name == "notebooks" and cwd.parent.name == "moe":
    MOE_ROOT = cwd.parent
elif (cwd / "dynamic_quality_aware_classwise_aggregation").exists():
    MOE_ROOT = cwd / "dynamic_quality_aware_classwise_aggregation" / "scene_daynight_dqa" / "moe"
else:
    MOE_ROOT = cwd

SCENE_ROOT = MOE_ROOT.parent
WORKSPACE = MOE_ROOT / "output" / "07_non_residual_moe_theory_loops"
SOURCE_WORKSPACE = SCENE_ROOT / "output" / "03_main_bn_residual_dqa_experiment"
RUNNER = MOE_ROOT / "scripts" / "run_moe_07_non_residual_moe_theory_loops.py"

print("MOE_ROOT", MOE_ROOT)
print("WORKSPACE", WORKSPACE)
print("SOURCE_WORKSPACE", SOURCE_WORKSPACE)
print("RUNNER", RUNNER)

MOE_ROOT /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe
WORKSPACE /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/output/07_non_residual_moe_theory_loops
SOURCE_WORKSPACE /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/03_main_bn_residual_dqa_experiment
RUNNER /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/scripts/run_moe_07_non_residual_moe_theory_loops.py


## Execute Fifteen Non-residual Loops

In [2]:
cmd = [
    sys.executable,
    str(RUNNER),
    "--workspace-root", str(WORKSPACE),
    "--source-workspace", str(SOURCE_WORKSPACE),
    "--notify",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=MOE_ROOT, check=True)

/opt/venv/bin/python3 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/scripts/run_moe_07_non_residual_moe_theory_loops.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/output/07_non_residual_moe_theory_loops --source-workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/03_main_bn_residual_dqa_experiment --notify


DiscordNotifyResult(ok=True, chunks_sent=1, status_codes=(204,), dry_run=False, error=None)
Top 5 non-residual MoE theory loops:
loop03_expert_choice_box_buckets 0.206262 high Expert Choice Routing
loop02_base_global_balanced_assignment 0.205962 medium BASE Layers
loop01_loss_free_balanced_pseudogt_router 0.205662 medium Auxiliary-Loss-Free Load Balancing / DeepSeek-V3
loop05_mixture_of_depths_pseudogt_compute 0.205267 medium Mixture-of-Depths
loop04_grin_gradient_informed_router 0.204713 medium GRIN / SparseMixer-v2
DiscordNotifyResult(ok=True, chunks_sent=1, status_codes=(204,), dry_run=False, error=None)


CompletedProcess(args=['/opt/venv/bin/python3', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/scripts/run_moe_07_non_residual_moe_theory_loops.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/output/07_non_residual_moe_theory_loops', '--source-workspace', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/03_main_bn_residual_dqa_experiment', '--notify'], returncode=0)

## Scoreboard

In [3]:
scoreboard = pd.read_csv(WORKSPACE / "stats" / "07_non_residual_moe_theory_scoreboard.csv")
display(
    scoreboard[
        [
            "loop_id",
            "paper_seed",
            "screened_projected_map50_95",
            "screened_delta_map50_95",
            "rank_score",
            "confidence",
            "dqa_translation",
            "rationale",
        ]
    ]
)

,loop_id,paper_seed,screened_projected_map50_95,screened_delta_map50_95,rank_score,confidence,dqa_translation,rationale
0,loop03_expert_choice_box_buckets,Expert Choice Routing,0.206262,0.006262,0.208762,high,各expertがclass/scale/density bucketから固定数のpseudo...,targets pseudoGT collapse signal 0.0030; uses ...
1,loop02_base_global_balanced_assignment,BASE Layers,0.205962,0.005962,0.208462,medium,roundごとにpseudo boxを全体で見て、expert容量制約つきの割当問題として選...,targets pseudoGT collapse signal 0.0030; uses ...
2,loop01_loss_free_balanced_pseudogt_router,Auxiliary-Loss-Free Load Balancing / DeepSeek-V3,0.205662,0.005662,0.208162,medium,各pseudo boxをexpertに割り当てる前に、直近のexpert利用率からrouti...,targets pseudoGT collapse signal 0.0030; uses ...
3,loop05_mixture_of_depths_pseudogt_compute,Mixture-of-Depths,0.205267,0.005267,0.207767,medium,easy/stable boxは軽く、境界/夜/小物体は追加augmentationや追加t...,targets pseudoGT collapse signal 0.0030; alloc...
4,loop04_grin_gradient_informed_router,GRIN / SparseMixer-v2,0.204713,0.004713,0.207213,medium,小さいprobe batchでhead/neckのgradient agreementを測り...,targets pseudoGT collapse signal 0.0030; uses ...
5,loop06_router_tuning_only,Router-Tuning / MindSkip,0.204225,0.004225,0.206725,medium,YOLO重みは固定し、pseudo box assignment/router parame...,targets pseudoGT collapse signal 0.0030; delay...
6,loop10_softmoe_slot_pseudogt_pooling,Soft MoE,0.203962,0.003962,0.205162,medium,box単位ではなく、複数boxのweighted slotをpseudoGT trainin...,targets pseudoGT collapse signal 0.0030; uses ...
7,loop11_vmoe_adaptive_per_image_compute,V-MoE adaptive compute,0.203391,0.003391,0.204591,low,画像ごとにtop-k expert数を変え、day/easyはtop1、night/dens...,uses load/density imbalance 0.0011; allocates ...
8,loop07_deepseek_fine_grained_expert_segmentation,DeepSeekMoE,0.203149,0.003149,0.204349,low,scene×time×class-density×scaleを小さいmicro-expert...,uses load/density imbalance 0.0011; adds fine-...
9,loop15_causal_factor_router,Mixture of Causal Experts / domain causal MoE,0.204325,0.004325,0.204325,medium,sceneやday/nightを直接expert名にせず、介入可能な因子としてrouter特...,targets pseudoGT collapse signal 0.0030; facto...


## Fifteen-loop Trace

In [4]:
trace = pd.read_csv(WORKSPACE / "stats" / "07_non_residual_moe_theory_loop_trace.csv")
display(
    trace[
        [
            "loop_index",
            "loop_id",
            "step_1_research",
            "step_5_execution",
            "step_6_result_summary",
            "step_7_next_direction",
        ]
    ]
)

,loop_index,loop_id,step_1_research,step_5_execution,step_6_result_summary,step_7_next_direction
0,1,loop03_expert_choice_box_buckets,Expert Choice Routing,executed as fast non-residual MoE theory scree...,"projected mAP50:95=0.206262, delta=0.006262, c...",promote to full design notebook
1,2,loop02_base_global_balanced_assignment,BASE Layers,executed as fast non-residual MoE theory scree...,"projected mAP50:95=0.205962, delta=0.005962, c...",keep as ablation candidate
2,3,loop01_loss_free_balanced_pseudogt_router,Auxiliary-Loss-Free Load Balancing / DeepSeek-V3,executed as fast non-residual MoE theory scree...,"projected mAP50:95=0.205662, delta=0.005662, c...",keep as ablation candidate
3,4,loop05_mixture_of_depths_pseudogt_compute,Mixture-of-Depths,executed as fast non-residual MoE theory scree...,"projected mAP50:95=0.205267, delta=0.005267, c...",keep as ablation candidate
4,5,loop04_grin_gradient_informed_router,GRIN / SparseMixer-v2,executed as fast non-residual MoE theory scree...,"projected mAP50:95=0.204713, delta=0.004713, c...",keep as ablation candidate
5,6,loop06_router_tuning_only,Router-Tuning / MindSkip,executed as fast non-residual MoE theory scree...,"projected mAP50:95=0.204225, delta=0.004225, c...",keep as ablation candidate
6,7,loop10_softmoe_slot_pseudogt_pooling,Soft MoE,executed as fast non-residual MoE theory scree...,"projected mAP50:95=0.203962, delta=0.003962, c...",keep as ablation candidate
7,8,loop11_vmoe_adaptive_per_image_compute,V-MoE adaptive compute,executed as fast non-residual MoE theory scree...,"projected mAP50:95=0.203391, delta=0.003391, c...",keep as ablation candidate
8,9,loop07_deepseek_fine_grained_expert_segmentation,DeepSeekMoE,executed as fast non-residual MoE theory scree...,"projected mAP50:95=0.203149, delta=0.003149, c...",keep as ablation candidate
9,10,loop15_causal_factor_router,Mixture of Causal Experts / domain causal MoE,executed as fast non-residual MoE theory scree...,"projected mAP50:95=0.204325, delta=0.004325, c...",keep as ablation candidate


## Selected Candidate

In [5]:
import json

selected_path = WORKSPACE / "stats" / "07_selected_non_residual_candidate.json"
selected = json.loads(selected_path.read_text(encoding="utf-8"))
print(json.dumps(selected, indent=2, ensure_ascii=False))

{
  "created_utc": "2026-05-08T23:19:33.160673+00:00",
  "protocol": "scene_daynight_dqa_moe_07_non_residual_theory_loops_v1",
  "selected_loop": "loop03_expert_choice_box_buckets",
  "selected_paper_seed": "Expert Choice Routing",
  "selected_hypothesis": "expert側が学習可能なboxを選ぶと、未学習expertと過学習expertの両方を避けられる。",
  "selected_dqa_translation": "各expertがclass/scale/density bucketから固定数のpseudo boxを選ぶ。",
  "selected_implementation_sketch": "expertごとにlearnability scoreを計算し、capacity factorつきでbox listを生成する。",
  "anchor": {
    "03_dqa_aggregate_map50_95": 0.2,
    "03_dqa_repair_map50_95": 0.195,
    "warmup_repair_map50_95": 0.186
  },
  "next_full_notebook_name": "05_loss_free_balanced_pseudogt_router_dqa",
  "non_residual_full_design": {
    "detector_update_policy": "do not compose checkpoint residuals; train/update via routed pseudoGT selection only",
    "router_state": "expert load bias updated each round from recent bucket usage",
    "pseudoGT_selection": "balanced assignment over class, 

## Markdown Report

In [6]:
report = WORKSPACE / "07_non_residual_moe_theory_report.md"
print(report)
print(report.read_text(encoding="utf-8")[:7000])

/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/moe/output/07_non_residual_moe_theory_loops/07_non_residual_moe_theory_report.md
# MoE x DQA 07: Non-residual MoE Theory Fifteen Loops

- created_utc: 2026-05-08T23:19:33.160794+00:00
- protocol: scene_daynight_dqa_moe_07_non_residual_theory_loops_v1
- mode: paper-driven hypothesis screening, not full detector training
- excluded family: checkpoint/residual composition

## Evidence Used

- 03 DQA aggregate mAP50:95: 0.200
- 03 DQA + repair mAP50:95: 0.195
- warmup + repair mAP50:95: 0.186
- pseudo score spread: 0.110
- pseudo density spread: 3.248
- day-night gap: 0.073

## Ranking

| rank | loop | rank score | projected mAP50:95 | delta | confidence | paper seed |
|---:|---|---:|---:|---:|---|---|
| 1 | loop03_expert_choice_box_buckets | 0.208762 | 0.206262 | 0.006262 | high | Expert Choice Routing |
| 2 | loop02_base_global_balanced_assignment | 0.208462 | 0.205962 | 0.005962 | medium | BASE Layers |